# HTML-парсинг и сохранение новостей

Ноутбук собирает 316 новостей с сайта `Lenta.ru`, преобразует их в таблицы `pandas`
и сохраняет четыре итоговых CSV-файла:

- `knowledge_base_100 news.csv`
- `knowledge_base_200 news.csv`
- `knowledge_base_300 news.csv`
- `test base_16.csv`

In [1]:
import time
from collections import defaultdict
from datetime import date, timedelta

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://lenta.ru"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/135.0.0.0 Safari/537.36"
    )
}

CATEGORY_MAP = {
    "Экономика": "economics",
    "Наука и техника": "science",
    "Культура": "culture",
    "Спорт": "sport",
}

CATEGORY_PATHS = {
    "economics": "/rubrics/economics/",
    "science": "/rubrics/science/",
    "culture": "/rubrics/culture/",
    "sport": "/rubrics/sport/",
}

CATEGORY_ORDER = ["economics", "science", "culture", "sport"]


def make_session():
    session = requests.Session()
    session.headers.update(HEADERS)
    return session


def normalize_text(text):
    return " ".join(str(text).replace("\xa0", " ").split())


def parse_archive_page(session, page_url):
    response = session.get(page_url, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    cards = soup.select("a.card-full-news._archive, a.card-mini-news._archive")
    records = []

    for card in cards:
        href = card.get("href")
        if not href or "/news/" not in href:
            continue

        rubric_node = card.select_one(".card-full-news__rubric, .card-mini-news__rubric")
        title_node = card.select_one(".card-full-news__title, .card-mini-news__title")
        if rubric_node is None or title_node is None:
            continue

        rubric = normalize_text(rubric_node.get_text(" ", strip=True))
        if rubric not in CATEGORY_MAP:
            continue

        full_url = f"{BASE_URL}{href}" if href.startswith("/") else href
        title = normalize_text(title_node.get_text(" ", strip=True))

        records.append(
            {
                "title": title,
                "url": full_url,
                "category": CATEGORY_MAP[rubric],
            }
        )

    return records


def extract_article(session, url, expected_category):
    response = session.get(url, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    title_node = soup.find("h1")
    rubric_node = soup.select_one("a.topic-header__rubric")
    content_node = soup.select_one("div.topic-body__content") or soup.select_one("div.topic-body")

    if title_node is None or rubric_node is None or content_node is None:
        return None

    rubric_href = rubric_node.get("href", "")
    if rubric_href != CATEGORY_PATHS[expected_category]:
        return None

    fragments = []
    for node in content_node.find_all(["p", "h2", "li"]):
        text = normalize_text(node.get_text(" ", strip=True))
        if text:
            fragments.append(text)

    if fragments:
        article_text = normalize_text(" ".join(fragments))
    else:
        article_text = normalize_text(content_node.get_text(" ", strip=True))

    if len(article_text) < 250:
        return None

    return {
        "title": normalize_text(title_node.get_text(" ", strip=True)),
        "text": article_text,
        "category": expected_category,
        "url": url,
    }


def collect_balanced_news(
    start_date=date.today(),
    target_per_category=79,
    lookback_days=180,
    max_pages_per_day=12,
    sleep_seconds=0.08,
):
    session = make_session()
    collected = {category: [] for category in CATEGORY_ORDER}
    seen_urls = set()

    for day_offset in range(lookback_days):
        current_day = start_date - timedelta(days=day_offset)
        day_url = current_day.strftime(f"{BASE_URL}/%Y/%m/%d/")
        day_added = defaultdict(int)

        for page_number in range(1, max_pages_per_day + 1):
            page_url = day_url if page_number == 1 else f"{day_url}page/{page_number}/"

            try:
                archive_records = parse_archive_page(session, page_url)
            except requests.HTTPError:
                break
            except requests.RequestException:
                continue

            if not archive_records:
                break

            for archive_record in archive_records:
                category = archive_record["category"]
                if len(collected[category]) >= target_per_category:
                    continue

                url = archive_record["url"]
                if url in seen_urls:
                    continue

                seen_urls.add(url)

                try:
                    article_record = extract_article(session, url, category)
                except requests.RequestException:
                    continue

                if article_record is None:
                    continue

                collected[category].append(article_record)
                day_added[category] += 1
                time.sleep(sleep_seconds)

                if all(len(collected[item]) >= target_per_category for item in CATEGORY_ORDER):
                    break

            if all(len(collected[item]) >= target_per_category for item in CATEGORY_ORDER):
                break

        print(
            current_day.isoformat(),
            {category: day_added[category] for category in CATEGORY_ORDER if day_added[category]},
            {category: len(collected[category]) for category in CATEGORY_ORDER},
        )

        if all(len(collected[item]) >= target_per_category for item in CATEGORY_ORDER):
            return collected

    raise RuntimeError("Недостаточно новостей для сборки итогового корпуса.")


def build_balanced_dataframe(collected, count_per_category, offset=0):
    rows = []
    for category in CATEGORY_ORDER:
        rows.extend(collected[category][offset:offset + count_per_category])

    frame = pd.DataFrame(rows, columns=["title", "text", "category", "url"])
    return frame.sample(frac=1, random_state=42).reset_index(drop=True)


def save_all_datasets(collected):
    knowledge_base_100 = build_balanced_dataframe(collected, count_per_category=25, offset=0)
    knowledge_base_200 = build_balanced_dataframe(collected, count_per_category=50, offset=0)
    knowledge_base_300 = build_balanced_dataframe(collected, count_per_category=75, offset=0)
    test_base_16 = build_balanced_dataframe(collected, count_per_category=4, offset=75)

    knowledge_base_100.to_csv("knowledge_base_100 news.csv", index=False, encoding="utf-8-sig")
    knowledge_base_200.to_csv("knowledge_base_200 news.csv", index=False, encoding="utf-8-sig")
    knowledge_base_300.to_csv("knowledge_base_300 news.csv", index=False, encoding="utf-8-sig")
    test_base_16.to_csv("test base_16.csv", index=False, encoding="utf-8-sig")

    return {
        "knowledge_base_100": knowledge_base_100,
        "knowledge_base_200": knowledge_base_200,
        "knowledge_base_300": knowledge_base_300,
        "test_base_16": test_base_16,
    }

In [2]:
print("СБОР И СОХРАНЕНИЕ НОВОСТЕЙ LENTA.RU")
print("План: 75 новостей на категорию для обучения и 4 новости на категорию для проверки.")

collected_news = collect_balanced_news()

for category in CATEGORY_ORDER:
    print(f"{category}: {len(collected_news[category])} новостей")

datasets = save_all_datasets(collected_news)

for dataset_name, dataset_frame in datasets.items():
    print("\n" + "=" * 80)
    print(dataset_name)
    print(dataset_frame["category"].value_counts().sort_index())
    print(dataset_frame.head(3))

СБОР И СОХРАНЕНИЕ НОВОСТЕЙ LENTA.RU
План: 75 новостей на категорию для обучения и 4 новости на категорию для проверки.
2026-04-21 {'economics': 47, 'science': 16, 'culture': 7, 'sport': 12} {'economics': 47, 'science': 16, 'culture': 7, 'sport': 12}
2026-04-20 {'economics': 32, 'science': 21, 'culture': 10, 'sport': 9} {'economics': 79, 'science': 37, 'culture': 17, 'sport': 21}
2026-04-19 {'science': 3, 'culture': 8, 'sport': 21} {'economics': 79, 'science': 40, 'culture': 25, 'sport': 42}
2026-04-18 {'science': 7, 'culture': 4, 'sport': 19} {'economics': 79, 'science': 47, 'culture': 29, 'sport': 61}
2026-04-17 {'science': 17, 'culture': 8, 'sport': 14} {'economics': 79, 'science': 64, 'culture': 37, 'sport': 75}
2026-04-16 {'science': 15, 'culture': 6, 'sport': 4} {'economics': 79, 'science': 79, 'culture': 43, 'sport': 79}
2026-04-15 {'culture': 9} {'economics': 79, 'science': 79, 'culture': 52, 'sport': 79}
2026-04-14 {'culture': 6} {'economics': 79, 'science': 79, 'culture': 58, 